# Tutorial 03: Agent Invocation in A2A Systems

A production-grade walkthrough of `A2A/03_agent_invocation`, focused on turning routing decisions into real tool execution through structured payloads and endpoint calls.

## 1) Where We Are in the Journey

In `02_ Agent Routing`, the system learned to pick an agent by matching user intent to discovered capabilities.
That solved selection, but not execution.
This tutorial exists to complete the loop: build MCP-style payloads and invoke the selected agent endpoint to produce outcomes.

## 2) What You Will Learn

- How invocation extends routing into full execution
- How `build_payload()` transforms natural language into tool arguments
- How `invoke()` uses discovered endpoint metadata for actual calls
- How `execute()` coordinates discovery, routing, payload building, and invocation
- Where this implementation is intentionally simple and where production hardening is needed

## 3) Why This Matters

In real A2A systems, value is created only when an intent reaches a concrete tool invocation and returns a result.
This step is the operational bridge between decision logic and business outcomes.
It also aligns tightly with MCP thinking: select capability, structure arguments, call tool endpoint, and normalize result.

## 4) Core Concept Deep Dive

### Intuition

Routing answers: "who should handle this?" Invocation answers: "what exact call should be sent, and what came back?"

### System design perspective

The target notebook composes a 5-stage pipeline:

1. Discover capabilities (`discover_capabilities`)
2. Select agent (`route`)
3. Build normalized tool payload (`build_payload`)
4. Execute remote call (`invoke`)
5. Orchestrate and trace (`execute`)

This division keeps each stage independently testable and lets you swap implementations (e.g., smarter planner, safer parser) without rewriting the whole workflow.

### Trade-offs in this implementation

- Pros: transparent flow, easy debugging, clear contract boundaries
- Cons: regex extraction is fragile, first-tool assumption is limiting, no retries/timeout policy in invoke
- Practical takeaway: this is an educational skeleton that exposes orchestration mechanics before optimization.

### Subtle notebook behavior

Some functions are defined more than once in `Untitled.ipynb`. In notebooks, the last executed definition wins, so execution order is part of system behavior.

## 5) Code Walkthrough (Only `A2A/03_agent_invocation`)

File in this step:

- `Untitled.ipynb`: end-to-end execution from query to invoked result

Key functions and their roles:

- `discover_capabilities()`: reads live agent cards and tool schemas
- `route(query, registry)`: selects an agent object by text matching
- `build_payload(query, agent)`: extracts numbers and maps to tool-specific args
- `invoke(agent, payload)`: posts JSON payload to selected agent endpoint
- `execute(query)`: composes all stages and prints intermediate trace

Dependency note:

The workflow depends on reusable contracts introduced in `00_Agents`: `/agent-info`, `/tools`, and `/invoke`.

In [1]:
# 6) Executable Cell: Setup
import requests
import re

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

print('Configured invocation targets:', AGENTS)

Configured invocation targets: ['http://localhost:8000', 'http://localhost:8001']


In [2]:
# 7) Executable Cell: Connectivity gate
def check_agents(agents):
    status = {}
    for url in agents:
        try:
            resp = requests.get(f"{url}/agent-info", timeout=2)
            if resp.status_code == 200:
                status[url] = 'UP (200)'
            else:
                status[url] = f'DOWN (HTTP {resp.status_code})'
        except Exception as e:
            status[url] = f'DOWN ({type(e).__name__})'
    return status

health = check_agents(AGENTS)
health

{'http://localhost:8000': 'DOWN (HTTP 404)',
 'http://localhost:8001': 'DOWN (ConnectionError)'}

In [ ]:
# 8) Executable Cell: Discovery + routing (aligned with target notebook)
def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            'agent': info['name'],
            'description': info['description'],
            'endpoint': info['endpoint'],
            'tools': tools['tools']
        })

    return registry

def route(query, registry):
    query = query.lower()

    for agent in registry:
        for tool in agent['tools']:
            tool_name = tool['name'].lower().replace('_', ' ')
            tool_desc = tool['description'].lower()
            if tool_name in query or any(word in query for word in tool_desc.split()):
                return agent

    return None

In [ ]:
# 9) Executable Cell: Payload builder + invoke
def build_payload(query, agent):
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))

    tool_name = agent['tools'][0]['name']

    if tool_name == 'calculate_interest':
        return {
            'tool': 'calculate_interest',
            'args': {
                'principal': numbers[0],
                'rate': numbers[1],
                'time': numbers[2]
            }
        }

    elif tool_name == 'add':
        return {
            'tool': 'add',
            'args': {
                'a': numbers[0],
                'b': numbers[1]
            }
        }

    return None

def invoke(agent, payload):
    response = requests.post(agent['endpoint'], json=payload)
    return response.json()

In [ ]:
# 10) Executable Cell: End-to-end execution
def execute(query):
    print('QUERY:', query)

    registry = discover_capabilities()
    agent = route(query, registry)

    if not agent:
        return 'No agent found'

    payload = build_payload(query, agent)
    print('SELECTED AGENT:', agent['agent'])
    print('PAYLOAD:', payload)

    result = invoke(agent, payload)
    print('RESULT:', result)

    return result

if all(v.startswith('UP') for v in health.values()):
    q1 = 'calculate interest for 1000 at 5 for 2 years'
    q2 = 'add 10 and 20'
    execute(q1)
    execute(q2)
else:
    print('Agents unavailable; start services and rerun from Cell 7.')

In [4]:
# 11) Executable Cell: Offline smoke test for parser + routing behavior
# Keep this cell self-contained if run independently.
if 'route' not in globals():
    def route(query, registry):
        query = query.lower()
        for agent in registry:
            for tool in agent['tools']:
                tool_name = tool['name'].lower().replace('_', ' ')
                tool_desc = tool['description'].lower()
                if tool_name in query or any(word in query for word in tool_desc.split()):
                    return agent
        return None

if 'build_payload' not in globals():
    def build_payload(query, agent):
        numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
        tool_name = agent['tools'][0]['name']
        if tool_name == 'calculate_interest':
            return {
                'tool': 'calculate_interest',
                'args': {
                    'principal': numbers[0],
                    'rate': numbers[1],
                    'time': numbers[2]
                }
            }
        if tool_name == 'add':
            return {
                'tool': 'add',
                'args': {
                    'a': numbers[0],
                    'b': numbers[1]
                }
            }
        return None

mock_registry = [
    {
        'agent': 'finance-agent',
        'endpoint': 'http://localhost:8001/invoke',
        'tools': [{'name': 'calculate_interest', 'description': 'Calculate simple interest'}]
    },
    {
        'agent': 'math-agent',
        'endpoint': 'http://localhost:8000/invoke',
        'tools': [{'name': 'add', 'description': 'Add two numbers'}]
    }
]

demo_query = 'calculate interest for 1000 at 5 for 2 years'
selected = route(demo_query, mock_registry)
print('Selected agent:', selected['agent'] if selected else None)
print('Built payload:', build_payload(demo_query, selected) if selected else None)

Selected agent: finance-agent
Built payload: {'tool': 'calculate_interest', 'args': {'principal': 1000.0, 'rate': 5.0, 'time': 2.0}}


## 6) System Flow Explanation

1. Agent metadata and tools are discovered from all registered URLs.
2. Query text is matched against discovered tool semantics to select one agent.
3. Query is transformed into a structured tool payload with explicit args.
4. Payload is sent to the selected agent's invocation endpoint.
5. Agent executes tool logic and returns structured JSON result.
6. Controller function (`execute`) coordinates traceability across all stages.

## 7) Important Concepts You Might Miss

- `build_payload()` assumes the first tool in the list is the one to call; this is a deliberate simplification.
- Regex number extraction can silently mis-parse natural language edge cases.
- Invocation is contract-sensitive: payload keys must match the target tool schema exactly.
- Returning full agent object from `route` enables direct endpoint access in `invoke`.
- Notebook cell execution order changes effective function definitions and runtime behavior.

## 8) Common Mistakes / Pitfalls

- Service ports and endpoint URLs drift from actual running agents.
- Query does not contain enough numeric values for payload construction.
- No explicit timeout/retry handling around POST invocation calls.
- Assuming route match implies payload correctness for every query shape.
- Ignoring error envelopes from agents and treating every response as successful.

## 9) Key Takeaways

- `03_agent_invocation` operationalizes A2A by executing selected capabilities.
- The core artifact is not only selection but a valid invocation payload and result.
- Clear stage boundaries make the workflow observable and easier to debug.
- Educational simplicity here highlights where production safeguards should be added next.
- This folder is the execution bridge between routing and multi-step workflows.

## 10) What’s Next (Strict Preview)

Next, `04_multi_agent_workflow` introduces chained execution across multiple agents in a single user request.
It solves the next problem: composing sequential steps and passing intermediate results between agents.
This matters because many real tasks require collaboration, not one isolated invocation.
You will move from single-hop execution to coordinated multi-step orchestration.